# 1. Machine Learning types and terminology

Learn to recognize supervised, unsupervised, semi-supervised, self-supervised, and reinforcement learning. Distinguish regression from classification, and accuracy from a training loss. Prerequisite: [What is Machine Learning?](01_what_is_machine_learning.ipynb). You need addition and division only. The short code examples repeat the same arithmetic so you can inspect it rather than trust a name.

## 2. What problem does this solve?

Before selecting an algorithm, describe the available information and desired decision. A shop with past purchase outcomes can learn a supervised predictor. A shop with customer measurements but no desired groups can explore clusters. A warehouse robot receives consequences after moving, so it must learn about actions and delayed rewards. Calling all three tasks “prediction” hides differences that determine evaluation.

The words supervised and unsupervised describe how learning information is supplied, not whether a human is present. People still choose data, representations, objectives, and constraints in every case.

## 3. Intuition and analogy

Supervised learning resembles practice questions with answer keys. Unsupervised learning resembles sorting unfamiliar objects without supplied categories. Semi-supervised learning has answer keys for only some questions. Self-supervised learning creates a task from the data itself, such as hiding a word and predicting it. Reinforcement learning resembles learning a game by taking actions and receiving rewards.

These are imperfect analogies. An answer key may be wrong; an automatically generated task may encourage shortcuts; a high game score may fail to represent what a user actually wants. The objective is a human design decision, not an inherent definition of success.

## 4. Terminology

**Features** describe each example. A **label** is the desired output supplied for supervised training. A **class** is a category such as spam or legitimate mail. A **continuous quantity** can, in principle, vary between nearby values, like travel time; regression also often models discrete counts as numerical outcomes. A **cluster** is a group inferred from similarity, not necessarily a known class.

A **state** describes an agent's situation; an **action** changes it; a **reward** evaluates a consequence. A **policy** chooses actions. **Training data** estimates parameters, **validation data** guides choices, and **test data** supports final evaluation after choices are fixed. These roles describe information use, not merely filenames.

## 5. Mathematical foundations

For $n$ examples and $p$ measured features, write $X\in\mathbb{R}^{n\times p}$. This means a rectangular table with $n$ rows and $p$ columns, containing real numbers. The symbol $\mathbb{R}$ denotes real numbers; no multiplication of measurements is implied by the shape. For one target per example, $y$ has shape $(n,)$. The row $x_i$ describes example $i$; $y_i$ is its target.

Classification accuracy is

$$\mathrm{accuracy}=\frac{\text{number of correct predictions}}{\text{number of evaluated predictions}}.$$

It is unitless, ranging from zero to one. Multiplying by 100 gives a percentage. Regression MAE averages absolute numerical errors and keeps the target's units. These metrics answer different questions; category labels such as 0 and 1 are codes, not distances between concepts.

## 6. Hand-calculated example

Four emails have true labels `[0, 1, 1, 0]`, where 1 means spam. Predictions `[0, 0, 1, 0]` are correct on emails one, three, and four. Accuracy is $3/4=0.75=75\%$. The second spam email was missed. If missed spam and blocked legitimate mail have different consequences, accuracy alone hides that difference.

For travel times 10 and 20 minutes, predictions 12 and 17 have absolute errors 2 and 3 minutes. MAE is 2.5 minutes. Asking whether a continuous prediction exactly equals the observed number would discard useful information about closeness.

## 7. Implementation from scratch

Python's `==` checks equality and produces `True` or `False`. In an arithmetic sum, these count as one and zero. `zip` pairs corresponding elements, so first predictions are compared with first labels. We assert equal lengths because `zip` otherwise stops at the shorter list.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
actual = [0, 1, 1, 0]
predicted = [0, 0, 1, 0]
assert len(actual) == len(predicted)
correct = [truth == guess for truth, guess in zip(actual, predicted)]
accuracy = sum(correct) / len(actual)
print('Correct for each email:', correct)
print('Accuracy:', accuracy)
assert accuracy == 0.75

## 8. Step-by-step output inspection

The list should be `[True, False, True, True]`. It preserves which example failed, whereas the average compresses all examples into one number. Always examine some individual mistakes before explaining an average score. The calculation evaluates supplied predictions; it does not train a classifier. A metric and a learning algorithm are different components.

## 9. Library implementation

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error

assert accuracy_score(actual, predicted) == accuracy
table = confusion_matrix(actual, predicted, labels=[0, 1])
print('Rows = true class; columns = predicted class\n', table)
assert table.tolist() == [[2, 0], [1, 1]]
assert mean_absolute_error([10, 20], [12, 17]) == 2.5

The confusion matrix has shape $(2,2)$. Row zero contains the two legitimate messages, both predicted legitimate. Row one contains two spam messages: one missed and one found. Explicit labels keep the row and column meaning fixed even if a small sample lacks one class.

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(table, cmap='Blues', vmin=0)
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(table[row, col]), ha='center', va='center', color='white' if table[row, col] > table.max() / 2 else 'black')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Legitimate', 'Spam'],
       yticklabels=['Legitimate', 'Spam'], xlabel='Predicted', ylabel='Actual',
       title='Counts reveal the missed spam message')
display(fig)
plt.close(fig)

The diagonal contains correct predictions. The lower-left cell is the missed spam message. A color alone can be ambiguous; written counts make this small plot interpretable.

## 11. A realistic experiment: rare outcomes

Imagine an invented screening sample with 99 ordinary transactions and one fraudulent transaction. Predicting every transaction ordinary achieves 99% accuracy while finding no fraud. This example illustrates a measurement failure, not a fraud detection system.

In [ ]:
rare_actual = np.array([0] * 99 + [1])
all_ordinary = np.zeros(100, dtype=int)
print('Accuracy:', accuracy_score(rare_actual, all_ordinary))
frauds_found = np.sum((rare_actual == 1) & (all_ordinary == 1))
fraud_recall = frauds_found / np.sum(rare_actual == 1)
print('Fraction of actual frauds detected:', fraud_recall)
assert accuracy_score(rare_actual, all_ordinary) == 0.99
assert fraud_recall == 0

Recall measures the fraction of actual positives detected. Here a positive means fraud; “positive” does not mean desirable. Before modeling, agree on which category matters and which errors are costly.

## 12. Choices before hyperparameters

We have no trained estimator in this lesson, so no learned parameters or algorithm hyperparameters. We choose task formulation, label meaning, and evaluation metric. Later, a probability threshold such as 0.5 will turn a probability into a category. That threshold is chosen by us; it should reflect validation evidence and error costs rather than convention alone.

## 13. Evaluation across learning types

For supervised learning, compare predictions with held-out labels. For clustering, inspect stability and usefulness; a compact cluster is not automatically meaningful. Semi-supervised learning still needs an independent labeled evaluation sample. Self-supervised training loss measures its constructed task, while downstream evaluation tests the skill we actually want. Reinforcement learning evaluates return over episodes, ideally with separate random seeds and environments.

A reward can be delayed: a move now may affect success several moves later. A labeled example usually supplies an outcome without requiring the learner to choose the process that generated it. This is why reinforcement learning is not just classification with a different word for labels.

## 14. Assumptions and limitations

Labels can reflect inconsistent human decisions or biased historical processes. Unlabeled does not mean unbiased. A dataset can mix protocols: a pretrained self-supervised model can later receive supervised fine-tuning. Categories are useful descriptions of learning signals, not sealed boxes into which every real system fits exactly once.

## 15. Mistakes and debugging

Do not treat integer category codes as quantities with meaningful spacing. Do not name a clustering result “high risk” without independent evidence. Do not call 99% accuracy excellent before checking the class distribution. Check which axis of a confusion matrix means actual labels; different plotting conventions can otherwise reverse your interpretation.

## 16. Choosing among alternatives

If the target is tomorrow's price, begin with regression. If it is whether a payment fails, begin with classification. If the goal is exploring groups without supplied outcomes, consider clustering. If decisions change future information and rewards, consider an interactive decision framework. Sometimes a clear business rule and a spreadsheet are adequate; algorithm selection follows problem definition.

## 17. Exercises

1. **Beginner:** Classify predicting temperature as regression or classification and explain the unit.
2. **Beginner:** Classify grouping songs without genre labels by learning type.
3. **Beginner:** Calculate accuracy for truths `[1,0,1]` and predictions `[1,1,1]`.
4. **Intermediate:** Explain how hiding a word supplies a self-supervised target without human annotation.
5. **Intermediate:** Design two metrics for the rare-fraud example and explain why accuracy is insufficient.
6. **Challenge:** A recommender's suggestions change what users see and click. Explain why evaluating only historical clicks can mislead, and propose a cautious evaluation design.

## 18. Detailed solutions

1. Regression predicts a numerical temperature, such as degrees Celsius. Converting it to “hot/not hot” would create a different classification task.
2. This is unsupervised clustering. The resulting groups need not match musical genres people would assign.
3. Two of three are correct, so accuracy is $2/3$, approximately 66.7%. The false positive is the middle example.
4. The original sentence already contains the missing word. Hiding it creates an input-target pair, but evaluation must still separate documents to avoid memorization.
5. Use fraud recall to measure found frauds and precision to measure the fraction of flagged transactions that are actually fraudulent. Reporting a false-positive count also helps estimate review workload. Precision is undefined when nothing is flagged unless a convention is stated.
6. Unshown items cannot receive clicks, so the log reflects the old recommendation policy as well as user interests. Start with chronological offline evaluation and exposure-aware limitations, then consider a monitored randomized experiment with consent and guardrails where appropriate. Do not claim offline accuracy proves an intervention benefits users.

## 19. Knowledge check

**Are clusters labels?** They are inferred groups, not supplied truth. **Can regression predict counts?** Yes, though suitable count models may be needed. **Does self-supervision remove human choices?** No; the task and data are still designed. **Why validation and test sets?** Selection and final assessment have different information roles. **Is reward the same as correctness?** No; reward is a designed consequence signal that may be delayed or incomplete.

## 20. Interview preparation

**Supervised versus unsupervised?** Supplied prediction targets versus structure inferred without them. **Classification versus regression?** Categories versus numerical outcomes, with different appropriate metrics. **Why can accuracy mislead?** Unequal class prevalence and error costs can hide failure on the important group. **Semi-supervised versus self-supervised?** The former combines labeled and unlabeled examples; the latter constructs targets from data. **What makes RL difficult?** Actions affect future states and available evidence, and delayed rewards complicate credit assignment.

## 21. Summary and next steps

Begin with the learning signal and decision, then choose metrics and models. Continue to [Python and Jupyter introduction](03_python_and_jupyter_introduction.ipynb), where we slow down and learn how the code itself works.